# <span style="color:#cc416d">Reproducir EXACTAMENTE la densificación del plugin</span>

In [15]:
import math
from qgis.core import (
    QgsApplication,
    QgsVectorLayer,
    QgsRasterLayer,
    QgsPointXY, QgsGeometry
)


### <span style="color:#cc416d">1. Carga de datos</span>

In [3]:
ruta_dem = r"C:\Proyectos\2026\seccion\dem2.tif"

ruta_linea_perfil = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\ejemplo_seccion3_guia.shp"
)


### <span style="color:#cc416d">2. Verificación de CRS</span>

In [4]:
dem = QgsRasterLayer(ruta_dem, "DEM")
linea_layer = QgsVectorLayer(ruta_linea_perfil, "seccion", "ogr")

print("DEM válido:", dem.isValid())
print("Línea válida:", linea_layer.isValid())
print("CRS DEM:", dem.crs().authid())
print("CRS línea:", linea_layer.crs().authid())
print("Tamaño píxel X:", dem.rasterUnitsPerPixelX())
print("Tamaño píxel Y:", dem.rasterUnitsPerPixelY())
print("Extent DEM:", dem.extent().toString())

DEM válido: True
Línea válida: True
CRS DEM: EPSG:6368
CRS línea: EPSG:6368
Tamaño píxel X: 5.0
Tamaño píxel Y: 5.0
Extent DEM: 616700.0000000000000000,2107955.0000000000000000 : 622915.0000000000000000,2115230.0000000000000000


In [5]:
feat_linea = next(linea_layer.getFeatures())
geom_linea = feat_linea.geometry()

print("Longitud:", geom_linea.length())
print("Vértices originales:", len(list(geom_linea.vertices())))

Longitud: 5772.659572948623
Vértices originales: 2


### <span style="color:#cc416d">3. función igual a la del plugin</span>

In [8]:
DISTANCIA = 5.0       # igual al tamaño del píxel

geom_densificada = geom_linea.densifyByDistance(DISTANCIA)

vertices = list(geom_densificada.vertices())

print("Vértices densificados:", len(vertices))

Vértices densificados: 1156


### <span style="color:#cc416d">3.1 inspeccionar los primeros punto</span>

In [9]:
for i, pt in enumerate(vertices[:10]):
    print(
        i,
        round(pt.x(),3),
        round(pt.y(),3)
    )

0 617150.224 2113765.924
1 617155.074 2113764.715
2 617159.923 2113763.505
3 617164.772 2113762.296
4 617169.622 2113761.086
5 617174.471 2113759.877
6 617179.321 2113758.667
7 617184.17 2113757.458
8 617189.02 2113756.248
9 617193.869 2113755.039


### <span style="color:#cc416d">3.2 inspeccionar los últimos punto</span>

In [10]:
for i, pt in enumerate(vertices[-10:]):
    print(
        len(vertices)-10+i,
        round(pt.x(),3),
        round(pt.y(),3)
    )

1146 622707.658 2112379.841
1147 622712.507 2112378.632
1148 622717.356 2112377.422
1149 622722.206 2112376.213
1150 622727.055 2112375.003
1151 622731.905 2112373.794
1152 622736.754 2112372.584
1153 622741.604 2112371.375
1154 622746.453 2112370.165
1155 622751.302 2112368.956


# <span style="color:#cc416d">Etapa 1.2</span>
### <span style="color:#cc416d">Comprobar la separación REAL entre vértices</span>

In [14]:
distancias=[]

for i in range(1,len(vertices)):

    dx=vertices[i].x()-vertices[i-1].x()
    dy=vertices[i].y()-vertices[i-1].y()

    distancias.append(
        math.sqrt(dx*dx+dy*dy)
    )

print("mín =",min(distancias))
print("máx =",max(distancias))
print("prom =",sum(distancias)/len(distancias))

mín = 4.997973656049072
máx = 4.997973656274716
prom = 4.997973656232574


La densificación no está generando el desfase

La línea tiene:
<p>
Longitud: 5772.659572948623 m
Vértices densificados: 1156
Segmentos entre vértices: 1155
</p>

La separación teórica es:

5772.659572948623/1155= 4.9979736562 m

Eso coincide exactamente con los resultados:

- mínima: 4.997973656049072
- máxima: 4.997973656274716
- promedio: 4.997973656232574
  
<p>
Es decir, densifyByDistance(5.0) no coloca necesariamente un punto exactamente cada 5.000 m. Divide la línea en un número entero de segmentos iguales, asegurando que ninguno supere los 5 m.
</p>
En este caso:

1155×4.9979736562=5772.6595729 m

Por eso aparecen 1156 vértices, contando el inicial y el final.

Qué demuestran las coordenadas

Los primeros puntos avanzan aproximadamente:

ΔX = 4.849 m
ΔY = -1.209 m

La distancia es:

raiz cuadrada de (4.8492+(−1.209)2)   ≈4.998 m

Y los últimos puntos mantienen prácticamente el mismo desplazamiento. Esto indica que:
- la densificación comienza en el extremo correcto;
- termina en el extremo correcto;
- mantiene una separación uniforme;
- no introduce saltos;
- no desplaza lateralmente la línea;
- no altera su orientación.




# <span style="color:#cc416d">Etapa 1.3</span>
### <span style="color:#cc416d">Ahora debemos demostrar que la distancia acumulada asignada a cada punto coincide con la posición real del vértice sobre la línea.</span>

In [19]:

distancias_acumuladas = [0.0]

for i in range(1, len(vertices)):
    p_anterior = QgsPointXY(vertices[i - 1])
    p_actual = QgsPointXY(vertices[i])

    distancia_segmento = p_anterior.distance(p_actual)
    distancias_acumuladas.append(
        distancias_acumuladas[-1] + distancia_segmento
    )

print("Cantidad de vértices:", len(vertices))
print("Cantidad de distancias:", len(distancias_acumuladas))

print("\nPrimeras distancias acumuladas:")
for i in range(10):
    print(i, distancias_acumuladas[i])

print("\nÚltimas distancias acumuladas:")
for i in range(len(distancias_acumuladas) - 10, len(distancias_acumuladas)):
    print(i, distancias_acumuladas[i])

print("\nDistancia acumulada final:", distancias_acumuladas[-1])
print("Longitud geometría original:", geom_linea.length())
print(
    "Diferencia:",
    distancias_acumuladas[-1] - geom_linea.length()
)

Cantidad de vértices: 1156
Cantidad de distancias: 1156

Primeras distancias acumuladas:
0 0.0
1 4.997973656274716
2 9.995947312436478
3 14.993920968711194
4 19.99189462498591
5 24.989868281147672
6 29.987841937422388
7 34.985815593697104
8 39.983789249858866
9 44.98176290613358

Últimas distancias acumuladas:
1146 5727.677810042423
1147 5732.675783698697
1148 5737.673757354859
1149 5742.6717310111335
1150 5747.669704667408
1151 5752.66767832357
1152 5757.665651979844
1153 5762.663625636119
1154 5767.661599292281
1155 5772.659572948555

Distancia acumulada final: 5772.659572948555
Longitud geometría original: 5772.659572948623
Diferencia: -6.730260793119669e-11


In [20]:
errores_xy = []

for i, (vertice, distancia) in enumerate(
    zip(vertices, distancias_acumuladas)
):
    geom_interpolada = geom_linea.interpolate(distancia)

    if geom_interpolada.isEmpty():
        print("Geometría vacía en índice:", i)
        continue

    punto_interpolado = geom_interpolada.asPoint()

    dx = punto_interpolado.x() - vertice.x()
    dy = punto_interpolado.y() - vertice.y()

    error = (
        QgsPointXY(punto_interpolado).distance(
            QgsPointXY(vertice)
        )
    )

    errores_xy.append(error)

print("Error XY mínimo:", min(errores_xy))
print("Error XY máximo:", max(errores_xy))
print("Error XY promedio:", sum(errores_xy) / len(errores_xy))

Error XY mínimo: 0.0
Error XY máximo: 4.656612873077393e-10
Error XY promedio: 1.9637532661117897e-11


In [21]:
indices_revision = [
    0,
    1,
    2,
    100,
    500,
    1000,
    len(vertices) - 2,
    len(vertices) - 1
]

for i in indices_revision:
    vertice = vertices[i]
    distancia = distancias_acumuladas[i]

    punto_interpolado = (
        geom_linea
        .interpolate(distancia)
        .asPoint()
    )

    error = QgsPointXY(vertice).distance(
        QgsPointXY(punto_interpolado)
    )

    print(
        f"\nÍndice: {i}",
        f"\nDistancia perfil: {distancia:.12f}",
        f"\nDensificado:   {vertice.x():.6f}, {vertice.y():.6f}",
        f"\nInterpolado:   {punto_interpolado.x():.6f}, "
        f"{punto_interpolado.y():.6f}",
        f"\nError XY:      {error:.12f}"
    )


Índice: 0 
Distancia perfil: 0.000000000000 
Densificado:   617150.224168, 2113765.924293 
Interpolado:   617150.224168, 2113765.924293 
Error XY:      0.000000000000

Índice: 1 
Distancia perfil: 4.997973656275 
Densificado:   617155.073586, 2113764.714797 
Interpolado:   617155.073586, 2113764.714797 
Error XY:      0.000000000000

Índice: 2 
Distancia perfil: 9.995947312436 
Densificado:   617159.923004, 2113763.505300 
Interpolado:   617159.923004, 2113763.505300 
Error XY:      0.000000000000

Índice: 100 
Distancia perfil: 499.797365623181 
Densificado:   617635.166007, 2113644.974637 
Interpolado:   617635.166007, 2113644.974637 
Error XY:      0.000000000000

Índice: 500 
Distancia perfil: 2498.986828116237 
Densificado:   619574.933363, 2113161.176012 
Interpolado:   619574.933363, 2113161.176012 
Error XY:      0.000000000000

Índice: 1000 
Distancia perfil: 4997.973656232541 
Densificado:   621999.642559, 2112556.427731 
Interpolado:   621999.642559, 2112556.427731 
Error X

los errores están del órden 1e-10 m

Incluso en el último punto:

- Densificado:  622751.302410, 2112368.955764
- Interpolado:  622751.302410, 2112368.955764
- Error XY:     0.000000000116 m

Conclusión de esta parte de la etapa 1

Podemos descartar como origen del desfase:

- densifyByDistance(5.0)
- el número de vértices densificados;
- la separación entre vértices;
- el cálculo de la distancia acumulada;
- geom_linea.interpolate(distancia);
- la reconstrucción XY a partir de la distancia del perfil.


In [24]:
provider = dem.dataProvider()
resultados_muestreo = []

for i, (vertice, distancia) in enumerate(
    zip(vertices, distancias_acumuladas)
):
    valor, ok = provider.sample(
        QgsPointXY(vertice.x(), vertice.y()),
        1
    )

    es_nan = False

    if valor is not None:
        try:
            es_nan = math.isnan(valor)
        except TypeError:
            es_nan = False

    resultados_muestreo.append({
        "indice": i,
        "distancia": distancia,
        "x": vertice.x(),
        "y": vertice.y(),
        "elevacion": valor,
        "ok": ok,
        "nan": es_nan
    })

print("Total de vértices:", len(vertices))
print("Total de muestras:", len(resultados_muestreo))

print(
    "Muestras válidas:",
    sum(
        1 for r in resultados_muestreo
        if r["ok"] and not r["nan"]
    )
)

print(
    "Muestras con ok=False:",
    sum(
        1 for r in resultados_muestreo
        if not r["ok"]
    )
)

print(
    "Muestras NaN:",
    sum(
        1 for r in resultados_muestreo
        if r["nan"]
    )
)

Total de vértices: 1156
Total de muestras: 1156
Muestras válidas: 1156
Muestras con ok=False: 0
Muestras NaN: 0


In [ ]:
Después imprime las muestras inválidas:

In [25]:
invalidas = [
    r for r in resultados_muestreo
    if not r["ok"] or r["nan"]
]

print("Cantidad de muestras inválidas:", len(invalidas))

for r in invalidas[:20]:
    print(r)

Cantidad de muestras inválidas: 0


In [26]:
print("\nPrimeras muestras:")

for r in resultados_muestreo[:10]:
    print(
        r["indice"],
        f'dist={r["distancia"]:.6f}',
        f'xy=({r["x"]:.3f}, {r["y"]:.3f})',
        f'z={r["elevacion"]}',
        f'ok={r["ok"]}'
    )

print("\nÚltimas muestras:")

for r in resultados_muestreo[-10:]:
    print(
        r["indice"],
        f'dist={r["distancia"]:.6f}',
        f'xy=({r["x"]:.3f}, {r["y"]:.3f})',
        f'z={r["elevacion"]}',
        f'ok={r["ok"]}'
    )


Primeras muestras:
0 dist=0.000000 xy=(617150.224, 2113765.924) z=186.55999755859375 ok=True
1 dist=4.997974 xy=(617155.074, 2113764.715) z=186.41000366210938 ok=True
2 dist=9.995947 xy=(617159.923, 2113763.505) z=186.41000366210938 ok=True
3 dist=14.993921 xy=(617164.772, 2113762.296) z=186.85000610351562 ok=True
4 dist=19.991895 xy=(617169.622, 2113761.086) z=186.8000030517578 ok=True
5 dist=24.989868 xy=(617174.471, 2113759.877) z=186.22000122070312 ok=True
6 dist=29.987842 xy=(617179.321, 2113758.667) z=186.49000549316406 ok=True
7 dist=34.985816 xy=(617184.170, 2113757.458) z=187.0 ok=True
8 dist=39.983789 xy=(617189.020, 2113756.248) z=186.6300048828125 ok=True
9 dist=44.981763 xy=(617193.869, 2113755.039) z=187.11000061035156 ok=True

Últimas muestras:
1146 dist=5727.677810 xy=(622707.658, 2112379.841) z=696.3300170898438 ok=True
1147 dist=5732.675784 xy=(622712.507, 2112378.632) z=695.4099731445312 ok=True
1148 dist=5737.673757 xy=(622717.356, 2112377.422) z=695.489990234375 o